In [42]:
from sympy import Function, Symbol, symbols, simplify, Eq, Symbol, latex, pprint, collect, expand
from sympy import init_printing
from IPython.display import display, Math

init_printing(use_latex=True)

In [30]:
# ── time variable ─────────────────────────────────────────────────────────────
t = Symbol('t')

# ── delay parameters ──────────────────────────────────────────────────────────
tau12, tau21, tau13, tau31, tau23, tau32 = symbols(
    r'\tau_{12} \tau_{21} \tau_{13} \tau_{31} \tau_{23} \tau_{32}',
    real=True, positive=True
)

# ── laser angular frequencies (physical + measurement offsets) ────────────────
omega1, omega2, omega3 = symbols(r'\omega_1 \omega_2 \omega_3', real=True)
omega1m, omega2m, omega3m = symbols(r'\omega_1^m \omega_2^m \omega_3^m', real=True)

In [31]:
# ── abstract time-dependent functions ─────────────────────────────────────────
phi1  = Function(r'\phi_1')   # laser phase noise, s/c 1
phi2  = Function(r'\phi_2')
phi3  = Function(r'\phi_3')

epsilonA = Function(r'\epsilon_A')  # clock noise, s/c 1 (master)
epsilonB = Function(r'\epsilon_B')  # clock noise, s/c 2
epsilonC = Function(r'\epsilon_C')  # clock noise, s/c 3

q1 = Function('q_1')
q2 = Function('q_2')
q3 = Function('q_3')

In [32]:
def D(expr, tau):
    return expr.subs(t, t - tau)

In [33]:
# η (sci) definitions
eta12 = (D(phi2(t), tau12) - phi1(t) - (omega2 - omega1) * q1(t) + omega2 * (epsilonA(t) - D(epsilonA(t), tau12)))
eta21 = (D(phi1(t), tau21) - phi2(t) + omega1 * (epsilonB(t) - D(epsilonB(t), tau21)) - (omega1 - omega2) * q2(t))
eta13 = (D(phi3(t), tau13) - phi1(t) + omega3 * (epsilonA(t) - D(epsilonA(t), tau13)) - (omega3 - omega1) * q1(t))
eta31 = (D(phi1(t), tau31) - phi3(t) + omega1 * (epsilonC(t) - D(epsilonC(t), tau31)) - (omega1 - omega3) * q3(t))
eta23 = (D(phi3(t), tau23) - phi2(t) + omega3 * (epsilonB(t) - D(epsilonB(t), tau23)) - (omega3 - omega2) * q2(t))
eta32 = (D(phi2(t), tau32) - phi3(t) + omega2 * (epsilonC(t) - D(epsilonC(t), tau32)) - (omega2 - omega3) * q3(t))


# η (SB / sideband) definitions
eta12SB = simplify(D(phi2(t), tau12) - phi1(t) - (omega2 - omega1 + omega2m - omega1m) * q1(t) - omega1m * q1(t) + omega2m * D(q2(t), tau12) + (omega2 + omega2m) * (epsilonA(t) - D(epsilonA(t), tau12)))
eta21SB = simplify(D(phi1(t), tau21) - phi2(t) - (omega1 - omega2 + omega1m - omega2m) * q2(t) - omega2m * q2(t) + omega1m * D(q1(t), tau21) + (omega1 + omega1m) * (epsilonB(t) - D(epsilonB(t), tau21)))
eta13SB = simplify(D(phi3(t), tau13) - phi1(t)- (omega3 - omega1 + omega3m - omega1m) * q1(t)- omega1m * q1(t)+ omega3m * D(q3(t), tau13)+ (omega3 + omega3m) * (epsilonA(t) - D(epsilonA(t), tau13)))
eta31SB = simplify(D(phi1(t), tau31) - phi3(t)- (omega1 - omega3 + omega1m - omega3m) * q3(t)- omega3m * q3(t)+ omega1m * D(q1(t), tau31)+ (omega1 + omega1m) * (epsilonC(t) - D(epsilonC(t), tau31)))
eta23SB = simplify(D(phi3(t), tau23) - phi2(t)- (omega3 - omega2 + omega3m - omega2m) * q2(t)- omega2m * q2(t)+ omega3m * D(q3(t), tau23)+ (omega3 + omega3m) * (epsilonB(t) - D(epsilonB(t), tau23)))
eta32SB = simplify(D(phi2(t), tau32) - phi3(t)- (omega2 - omega3 + omega2m - omega3m) * q3(t)- omega3m * q3(t)+ omega2m * D(q2(t), tau32)+ (omega2 + omega2m) * (epsilonC(t) - D(epsilonC(t), tau32)))

In [ ]:
# Map spacecraft index → (phi, q, omega, omega_m, clock_epsilon)
sc = {
    1: (phi1, q1, omega1, omega1m, epsilonA),
    2: (phi2, q2, omega2, omega2m, epsilonB),
    3: (phi3, q3, omega3, omega3m, epsilonC),
}

tau = {
    (1,2): tau12, (2,1): tau21,
    (1,3): tau13, (3,1): tau31,
    (2,3): tau23, (3,2): tau32,
}

eta    = {}
etaSB  = {}
r = {}

include_phi = False  # set False to drop all φ terms
include_board_jitter = False  
include_clock_noise = True

for (i, j) in tau:
    phi_i, q_i, om_i, omm_i, eps_i = sc[i]
    phi_j, q_j, om_j, omm_j, eps_j = sc[j]
    t_ij = tau[(i, j)]

    phi_terms = (D(phi_j(t), t_ij) - phi_i(t)) if include_phi else 0
    board_terms_c = (om_j * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0
    board_terms_SB = ((om_j + omm_j) * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0
    clock_terms_C = (- (om_j - om_i) * q_i(t)) if include_clock_noise else 0
    clock_terms_SB = (- (om_j - om_i + omm_j - omm_i) * q_i(t) - omm_i * q_i(t) + omm_j * D(q_j(t), t_ij)) if include_clock_noise else 0

    eta[(i,j)] = (phi_terms + clock_terms_C + board_terms_c)

    etaSB[(i,j)] = collect(expand(phi_terms + clock_terms_SB + board_terms_SB), [q_i(t), q_j(t)])

    r[(i,j)] = simplify((eta[(i,j)] - etaSB[(i,j)]) / omm_j)

In [62]:
for (i, j) in tau:
    display(Math(r'\eta_{' + str(i) + str(j) + '} = ' + latex(eta[(i,j)])))
    display(Math(r'\eta_{' + str(i) + str(j) + '}^{SB} = ' + latex(etaSB[(i,j)])))
    display(Math(r'r_{' + str(i) + str(j) + '} = ' + latex(r[(i,j)])))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [63]:
X0 = collect(expand(eta[(1,2)] - eta[(1,3)] + D(eta[(2,1)], tau12) - D(eta[(3,1)], tau13)), [q_i(t), q_j(t), epsilonA(t), epsilonB(t), epsilonC(t)])
display(Math(r'X_0 = ' + latex(X0)))

<IPython.core.display.Math object>